## 1. ini files

In [ ]:
import configparser
config = configparser.ConfigParser()
config.read("config.ini")

print("database name:", config["database"]["db_name"])
print("timeout time in seconds:", config["settings"]["timeout"])
print("items:", config["database"]["items"])

database name: postgres
timeout time in seconds: 30
items: <class 'str'>


## 2. XML files

In [17]:
import xml.etree.ElementTree as ET
tree = ET.parse("config.xml")
root = tree.getroot()

# To access a particular key, you need to navigate through the “root” e.g. to get the port:
print("database port:", root.find("./database/db_port").text)
print("time out time in seconds:", root.find("./settings/timeout/number").text)

database port: 5432
time out time in seconds: 30


## 3. JSON files

In [ ]:
import json
with open("config.json", "r") as f:
    config = json.load(f)

config

{'database': {'db_name': 'postgres',
  'db_port': 5432,
  'db_user': 'admin',
  'db_password': 'secret',
  'db_departments': ['Finance', 'HR']},
 'settings': {'debug': True, 'timeout': {'unit': 'seconds', 'number': 30}}}

## 4. YAML files

In [ ]:
import yaml

# data config example
with open("config.yaml", "r") as file:
    config = yaml.safe_load(file)

config

{'database': {'db_name': 'postgres',
  'db_port': 5432,
  'db_user': 'admin',
  'db_password': 'secret',
  'db_departments': ['Finance', 'HR']},
 'settings': {'debug': True, 'timeout': {'unit': 'seconds', 'number': 30}}}

In [ ]:
# pipeline config example
with open("pipeline_config.yaml", "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

config

{'data': {'path': 'attrition_data.csv',
  'loader': 'csv',
  'args': {'delimiter': ','}},
 'features': {'target': 'attrition',
  'numerical': ['age', 'dailyrate', 'distancefromhome'],
  'categorical': ['businesstravel', 'department', 'educationfield']},
 'preprocessing': {'scaler': {'type': 'standard', 'args': {}},
  'encoder': {'type': 'onehot',
   'args': {'drop': 'first',
    'sparse_output': False,
    'handle_unknown': 'ignore'}}},
 'model': {'type': 'logistic_regression',
  'args': {'C': 1.0, 'max_iter': 1000}},
 'evaluation': {'test_size': 0.2,
  'random_state': 42,
  'metrics': ['accuracy', 'f1', 'roc_auc']}}

## 6. Typed Python

In [28]:
from config import config

print(config.database.db_name)
print(config.settings.timeout)

postgres
30


# JSON -- Deep Dive

In [32]:
import json
with open("config.json", "r") as f:
    config_json = f.read()
    config = json.loads(config_json)

type(config_json), type(config)

(str, dict)

In [ ]:
cat = {
    "breed": "Siamese",
    "name": "Cookie",
    "age": 3,
    "gender": "male",
    "previous_owners": 2
}

# you can't write this to a file
# but you can extract attributes e.g. cat["age"]
print("The type of the 'cat' variable is:", type(cat))
print(cat)

# you can write this to a file
# but you can NOT extract the attributes
cat_json = json.dumps(cat)
print("The type of the 'cat_json' variable is:", type(cat_json))
print(cat_json)

The type of the 'cat' variable is: <class 'dict'>
{'breed': 'Siamese', 'name': 'Cookie', 'age': 3, 'gender': 'male', 'previous_owners': 2}
The type of the 'cat_json' variable is: <class 'str'>
{"breed": "Siamese", "name": "Cookie", "age": 3, "gender": "male", "previous_owners": 2}


In [ ]:
from pydantic import BaseModel

class PydanticCat(BaseModel):
    breed: str
    name: str
    age: int
    gender: str
    previous_owners: int

cat = PydanticCat(
    breed="Siamese",
    name="Cookie",
    age=3,
    gender="male",
    previous_owners=2
)

# you can't write this to a file
# but you can extract attributes e.g. cat["age"]
print(cat.model_dump(), type(cat.model_dump()))

# you can write this to a file
# but you can NOT extract the attributes
print(cat.model_dump_json(), type(cat.model_dump_json()))

{'breed': 'Siamese', 'name': 'Cookie', 'age': 3, 'gender': 'male', 'previous_owners': 2} <class 'dict'>
{"breed":"Siamese","name":"Cookie","age":3,"gender":"male","previous_owners":2} <class 'str'>


In [53]:
class Cat:
    def __init__(self, breed, name, previous_owners):
        self.breed = breed
        self.name = name
        self.previous_owners = previous_owners

    def to_json(self):
        return json.dumps({
            "breed": self.breed,
            "name": self.name,
            "previous_owners": self.previous_owners
        })
    
    @staticmethod
    def from_json(cat_json_string):
        cat_dict = json.loads(cat_json_string)
        relevant_keys = {k: v for k, v in cat_dict.items() if k in ["breed", "name", "previous_owners"]}
        return Cat(**relevant_keys)

cat = Cat(
    breed="Siamese",
    name="Cookie",
    previous_owners=2
)

# cat object of type Cat
print(cat, type(cat))

# convert Cat object to JSON in 2 methods:
# i. use .__dict__ to get a dictionary, then convert that dictionary into a JSON string or write to disk
# ii. create a custom .to_json() method
cat_json = cat.to_json()
print(cat_json, type(cat_json))
# you can now save this JSON to disk, send across the network or to a Document database or whatever

# let's save it to a disk
with open("cat.json", "w") as f:
    f.write(cat_json)

<__main__.Cat object at 0x000001BB60220AC0> <class '__main__.Cat'>
{"breed": "Siamese", "name": "Cookie", "previous_owners": 2} <class 'str'>


In [ ]:
# loader:
# you need to convert the JSON to a Cat object
with open("cat.json", "r") as f:
    cat_json = f.read()
cat = Cat.from_json(cat_json)
print(cat, type(cat))
# i. use json.load or json.loads to convert the JSON string into a dictionary, then give that dictionary to the constructor
# ii. create a static custom .from_json(json_string) method

# you now have an object to pass to your code!

<__main__.Cat object at 0x000001BB603F08E0> <class '__main__.Cat'>
